# 📄 Layout Detection (OpenDataLoader) + OCR (EasyOCR)

### Pipeline
```
PDF → OpenDataLoader JSON (kids tree) → flatten all elements → filter TEXT types
    → crop bbox from rendered page → EasyOCR → display text
```
OpenDataLoader JSON schema uses **`kids`** as the root array (not `elements`).  
Elements are **nested**: `text block → kids`, `table → rows → cells → kids`, `list → list items → kids`.  
This notebook correctly flattens the whole tree before filtering.

> Enable GPU: **Runtime → Change runtime type → T4 GPU**

## Step 1 — Install Dependencies

In [ ]:
!apt-get install -y default-jdk -q
!pip install opendataloader-pdf easyocr pdf2image -q
!apt-get install -y poppler-utils -q
!java -version
print('\n✅ Done!')

## Step 2 — Imports

In [ ]:
import json, random
import numpy as np
from pathlib import Path
from collections import Counter
from PIL import Image
from pdf2image import convert_from_path
import opendataloader_pdf
import easyocr
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from google.colab import files
import warnings; warnings.filterwarnings('ignore')
print('✅ Imports OK')

## Step 3 — Initialize EasyOCR

In [ ]:
ocr_reader = easyocr.Reader(['en'], gpu=True)
print('✅ EasyOCR ready')

## Step 4 — Upload PDFs

In [ ]:
uploaded = files.upload()
upload_dir = Path('/content/pdfs'); upload_dir.mkdir(exist_ok=True)
pdf_paths = []
for fname, content in uploaded.items():
    if not fname.lower().endswith('.pdf'):
        print(f'⚠️  Skipping {fname}'); continue
    p = upload_dir / fname
    p.write_bytes(content)
    pdf_paths.append(str(p))
    print(f'  ✅ {fname}  ({len(content)/1024:.1f} KB)')
print(f'\n{len(pdf_paths)} PDF(s) ready')

## Step 5 — Core Helpers
Key fix: the OpenDataLoader JSON root is `{..., "kids": [...]}`.  
Elements are nested trees — we recursively flatten them to get every leaf element with its bbox, type, and page number.

In [ ]:
OUTPUT_DIR = Path('/content/odl_output'); OUTPUT_DIR.mkdir(exist_ok=True)
DPI = 200

# ── ODL element types we want to OCR ──────────────────────────
TEXT_TYPES = {'paragraph', 'heading', 'caption', 'list item', 'text block'}
SKIP_TYPES = {'table', 'image', 'header', 'footer', 'table row', 'table cell'}

BBOX_COLOURS = {
    'paragraph':  'green',
    'heading':    'blue',
    'caption':    'orange',
    'list item':  'cyan',
    'list':       'cyan',
    'text block': 'green',
    'table':      'red',
    'image':      'brown',
    'header':     'gray',
    'footer':     'gray',
}


# ── 1. Run OpenDataLoader and load JSON ───────────────────────
def run_odl(pdf_path: str) -> dict:
    opendataloader_pdf.convert(
        input_path=[pdf_path],
        output_dir=str(OUTPUT_DIR),
        format=['json']
    )
    json_name = Path(pdf_path).stem + '.json'
    matches = list(OUTPUT_DIR.rglob(json_name))
    if not matches:
        raise FileNotFoundError(f'JSON not found for {pdf_path}')
    with open(matches[0], 'r', encoding='utf-8') as f:
        return json.load(f)


# ── 2. Recursively flatten the kids tree ─────────────────────
def flatten_elements(node_list: list, collected: list = None) -> list:
    """
    Walk the nested ODL element tree and collect every node that has
    a 'bounding box' and 'page number', regardless of nesting depth.
    Handles: kids, list items, rows→cells→kids.
    """
    if collected is None:
        collected = []
    for node in node_list:
        if not isinstance(node, dict):
            continue
        if 'bounding box' in node and 'page number' in node:
            collected.append(node)
        # recurse into all possible child list keys
        for child_key in ('kids', 'list items', 'rows'):
            if child_key in node and isinstance(node[child_key], list):
                flatten_elements(node[child_key], collected)
        # table rows → cells → kids
        if node.get('type') == 'table row' and 'cells' in node:
            flatten_elements(node['cells'], collected)
    return collected


# ── 3. Crop a bbox from a PIL page image ─────────────────────
def crop_bbox(page_img: Image.Image, bbox: list,
              page_w_pts: float, page_h_pts: float,
              padding: int = 4) -> Image.Image | None:
    """
    bbox = [left, bottom, right, top] in PDF points (origin = bottom-left).
    PIL origin = top-left, so we flip y.
    padding: extra pixels to add around crop (helps EasyOCR accuracy).
    """
    left, bottom, right, top = bbox
    W, H = page_img.size
    x0 = int((left  / page_w_pts) * W) - padding
    x1 = int((right / page_w_pts) * W) + padding
    y0 = int(((page_h_pts - top)    / page_h_pts) * H) - padding
    y1 = int(((page_h_pts - bottom) / page_h_pts) * H) + padding
    x0, x1 = max(0, x0), min(W, x1)
    y0, y1 = max(0, y0), min(H, y1)
    if x1 <= x0 or y1 <= y0:
        return None
    return page_img.crop((x0, y0, x1, y1))


# ── 4. Infer page dimensions from bboxes (fallback: A4) ───────
def infer_page_dims(elements: list) -> tuple:
    bboxes = [el['bounding box'] for el in elements if 'bounding box' in el]
    if bboxes:
        pw = max(b[2] for b in bboxes)
        ph = max(b[3] for b in bboxes)
        if pw > 100 and ph > 100:
            return pw, ph
    return 595.0, 842.0   # A4 fallback


# ── 5. Visualise bboxes on a page image ───────────────────────
def visualise_page(page_img, page_elements, pw, ph, title=''):
    W, H = page_img.size
    fig, ax = plt.subplots(1, 1, figsize=(11, 15))
    ax.imshow(page_img)
    ax.set_title(title, fontsize=11)
    ax.axis('off')

    for el in page_elements:
        if 'bounding box' not in el:
            continue
        el_type = el.get('type', 'unknown')
        color   = BBOX_COLOURS.get(el_type, 'gray')
        left, bottom, right, top = el['bounding box']
        x0 = (left  / pw) * W;  x1 = (right / pw) * W
        y0 = ((ph - top)    / ph) * H;  y1 = ((ph - bottom) / ph) * H
        rect = patches.Rectangle((x0, y0), x1-x0, y1-y0,
                                  linewidth=1.5, edgecolor=color, facecolor='none')
        ax.add_patch(rect)
        ax.text(x0, max(y0-3, 0), el_type, fontsize=6, color=color, clip_on=True)

    handles = [patches.Patch(color=c, label=l) for l, c in BBOX_COLOURS.items()]
    ax.legend(handles=handles, loc='upper right', fontsize=7, framealpha=0.8)
    plt.tight_layout(); plt.show()


# ── 6. OCR text elements on one page ──────────────────────────
def ocr_page(page_img, text_elements, pw, ph):
    # Sort top → bottom (PDF: higher 'top' value = higher on page)
    text_elements = sorted(text_elements, key=lambda el: -el['bounding box'][3])
    lines = []
    for el in text_elements:
        crop = crop_bbox(page_img, el['bounding box'], pw, ph)
        if crop is None:
            continue
        result = ocr_reader.readtext(np.array(crop), detail=0)
        text = ' '.join(result).strip()
        if text:
            lines.append((el.get('type', ''), text))
    return lines


print('✅ Helpers defined')

## Step 6 — Process Each PDF: Layout Detection + Bbox Visualisation + OCR

Specific pages per PDF (edit `PAGE_MAP` if needed):
- `Researchpaper_KAI` → page **6**
- `apple 10k KAI` → page **38**
- `Laptopcatalogue_KAI` → page **5**
- Others → random page

In [ ]:
# ── Set target pages (stem match, case-insensitive) ───────────
PAGE_MAP = {
    'researchpaper_kai':    6,
    'apple 10k kai':        38,
    'laptopcatalogue_kai':  5,
}  # anything not listed gets a random page


for pdf_path in pdf_paths:
    filename = Path(pdf_path).name
    stem_lower = Path(pdf_path).stem.lower()

    print(f'\n{"="*65}')
    print(f'📄  PDF : {filename}')

    # ── A. Run OpenDataLoader ──────────────────────────────────
    print('    Running OpenDataLoader...', end=' ')
    odl_json = run_odl(pdf_path)
    total_pages = odl_json.get('number of pages', '?')
    print(f'done  ({total_pages} pages)')

    # ── B. Flatten the nested kids tree ───────────────────────
    root_kids = odl_json.get('kids', [])
    all_elements = flatten_elements(root_kids)
    print(f'    Total elements (flattened): {len(all_elements)}')

    if not all_elements:
        print('    ⚠️  Still 0 elements after flattening — check JSON manually')
        # Print raw JSON structure for debugging
        print(f'    Raw JSON top-level keys: {list(odl_json.keys())}')
        if root_kids:
            print(f'    First kid keys: {list(root_kids[0].keys()) if root_kids else "empty"}')
        continue

    # ── C. Group by page ──────────────────────────────────────
    pages_dict = {}
    for el in all_elements:
        pg = el.get('page number', 1)
        pages_dict.setdefault(pg, []).append(el)

    # Element type breakdown
    type_counts = Counter(el.get('type', 'unknown') for el in all_elements)
    print(f'    Element types (doc-wide): {dict(type_counts)}')

    # ── D. Choose page ────────────────────────────────────────
    target_pg = PAGE_MAP.get(stem_lower)
    if target_pg is None or target_pg not in pages_dict:
        target_pg = random.choice(list(pages_dict.keys()))
        print(f'    (Random page selected)')
    print(f'    Target page : {target_pg}')

    page_els = pages_dict[target_pg]
    page_type_counts = Counter(el.get('type','?') for el in page_els)
    print(f'    Elements on page {target_pg}: {dict(page_type_counts)}')

    # ── E. Render page image ──────────────────────────────────
    print(f'    Rendering page {target_pg} at {DPI} DPI...')
    page_imgs = convert_from_path(
        pdf_path, dpi=DPI, first_page=target_pg, last_page=target_pg
    )
    page_img = page_imgs[0]
    pw, ph = infer_page_dims(page_els)

    # ── F. Visualise ALL bboxes on the page ───────────────────
    print(f'    Visualising layout detections...')
    visualise_page(
        page_img, page_els, pw, ph,
        title=f'{filename}  |  Page {target_pg}  — OpenDataLoader Layout'
    )

    # ── G. Filter to text-type elements only ──────────────────
    text_els = [
        el for el in page_els
        if el.get('type','').lower() in TEXT_TYPES
    ]
    skip_els = [
        el for el in page_els
        if el.get('type','').lower() in SKIP_TYPES
    ]
    print(f'    Text elements to OCR : {len(text_els)}')
    print(f'    Skipped (table/image): {len(skip_els)}')

    # ── H. EasyOCR on each text crop ──────────────────────────
    print(f'    Running EasyOCR...\n')
    print(f'{'─'*65}')
    print(f'  EXTRACTED TEXT | {filename} | Page {target_pg}')
    print(f'{'─'*65}')

    if not text_els:
        print('  [No text-type elements on this page]')
    else:
        ocr_lines = ocr_page(page_img, text_els, pw, ph)
        if not ocr_lines:
            print('  [EasyOCR returned no text — try increasing DPI]')
        for el_type, text in ocr_lines:
            prefix = '### ' if 'heading' in el_type else ''
            print(prefix + text)

    print(f'{'─'*65}')

print('\n✅ Done!')

## Step 7 — Debug: Inspect Raw JSON Structure
Run this if Step 6 still shows 0 elements. It prints the raw JSON so we can see exactly what ODL returned.

In [ ]:
# ── Inspect raw JSON of any one PDF ───────────────────────────
INSPECT_PDF = pdf_paths[0]   # change index as needed

odl_raw = run_odl(INSPECT_PDF)

print('Top-level keys:', list(odl_raw.keys()))
print('number of pages:', odl_raw.get('number of pages'))

kids = odl_raw.get('kids', [])
print(f'\nkids count: {len(kids)}')

if kids:
    print('\nFirst kid (pretty):')
    print(json.dumps(kids[0], indent=2)[:2000])   # first 2000 chars
else:
    print('kids is EMPTY — printing full JSON (first 3000 chars):')
    print(json.dumps(odl_raw, indent=2)[:3000])

## Step 8 — Re-run on Any Specific Page

In [ ]:
TARGET_PDF_IDX = 0   # index into pdf_paths
TARGET_PAGE    = 1   # 1-based

pdf_path2 = pdf_paths[TARGET_PDF_IDX]
fname2    = Path(pdf_path2).name

odl2      = run_odl(pdf_path2)
all_els2  = flatten_elements(odl2.get('kids', []))
page_els2 = [el for el in all_els2 if el.get('page number') == TARGET_PAGE]

assert page_els2, f'No elements found on page {TARGET_PAGE}. Pages available: {sorted(set(e["page number"] for e in all_els2))}'

imgs2 = convert_from_path(pdf_path2, dpi=DPI, first_page=TARGET_PAGE, last_page=TARGET_PAGE)
img2  = imgs2[0]
pw2, ph2 = infer_page_dims(page_els2)

visualise_page(img2, page_els2, pw2, ph2,
               title=f'{fname2}  |  Page {TARGET_PAGE}')

text_els2 = [el for el in page_els2 if el.get('type','').lower() in TEXT_TYPES]
print(f'\n--- EXTRACTED TEXT | {fname2} | Page {TARGET_PAGE} ---\n')
for el_type, text in ocr_page(img2, text_els2, pw2, ph2):
    print(('### ' if 'heading' in el_type else '') + text)
print(f'\n--- END | {len(text_els2)} text elements ---')